# Tidying `messy_survey_wide_50k.csv`

Plan: one **observational unit per table**.

| Tidy table | One row per | Built from |
|---|---|---|
| `respondents` | respondent | id, province, interview date, income bracket |
| `weekly_spending` | respondent x week | `wk1` ... `wk26` |
| `satisfaction` | respondent x survey item | `q1_a` ... `q1_e` |
| `household_members` | respondent x sex x age group | `hh_<sex>_<age>` columns |

In [1]:
import pandas as pd

df = pd.read_csv("data/messy_survey_wide_50k.csv")
print("Original shape:", df.shape)

Original shape: (50000, 50)


## 1. Clean the dirty `province` values

In [2]:
print("Before:", sorted(df["province"].unique()))
df["province"] = df["province"].str.strip().str.upper()
print("After: ", sorted(df["province"].unique()))

Before: [' ON ', 'AB', 'BC', 'MB', 'NB', 'NL', 'NS', 'ON', 'PE', 'QC', 'SK', 'on']
After:  ['AB', 'BC', 'MB', 'NB', 'NL', 'NS', 'ON', 'PE', 'QC', 'SK']


## 2. Week number stored in headers: `wk1` ... `wk26`  ->  `week`, `spend`

In [3]:
wk_cols = [c for c in df.columns if c.startswith("wk")]

weekly_spending = df.melt(
    id_vars="respondent_id",
    value_vars=wk_cols,
    var_name="week",
    value_name="spend",
).dropna(subset=["spend"])

weekly_spending["week"] = weekly_spending["week"].str.replace("wk", "").astype(int)
weekly_spending = weekly_spending.sort_values(["respondent_id", "week"]).reset_index(drop=True)
print(weekly_spending.shape)
weekly_spending.head()

(779314, 3)


,respondent_id,week,spend
0,1000001,1,72.35
1,1000001,3,76.40
2,1000001,5,160.98
3,1000001,6,293.60
4,1000001,7,117.93


## 3. Survey battery items in headers: `q1_a` ... `q1_e`  ->  `item`, `response`  (7 = refused)

In [4]:
q_cols = [c for c in df.columns if c.startswith("q1_")]

satisfaction = df.melt(
    id_vars="respondent_id",
    value_vars=q_cols,
    var_name="item",
    value_name="response",
).dropna(subset=["response"])

satisfaction["item"] = satisfaction["item"].str.replace("q1_", "")

# 7 is a refusal code, not a rating: keep that fact, but blank the rating
satisfaction["refused"] = satisfaction["response"] == 7
satisfaction.loc[satisfaction["refused"], "response"] = pd.NA
satisfaction["response"] = satisfaction["response"].astype("Int64")
satisfaction = satisfaction.sort_values(["respondent_id", "item"]).reset_index(drop=True)
print(satisfaction.shape)
satisfaction.head()

(237630, 4)


,respondent_id,item,response,refused
0,1000001,a,5,False
1,1000001,b,5,False
2,1000001,c,4,False
3,1000001,d,2,False
4,1000001,e,3,False


## 4. Income brackets stored as 0/1 flags: `inc_*`  ->  one `income` column

In [5]:
inc_cols = [c for c in df.columns if c.startswith("inc_")]

income = df.melt(
    id_vars="respondent_id",
    value_vars=inc_cols,
    var_name="income",
    value_name="flag",
)
income = income[income["flag"] == 1].drop(columns="flag")
income["income"] = income["income"].str.replace("inc_", "")

# Each respondent must have exactly one bracket
assert income["respondent_id"].is_unique and len(income) == len(df)

order = ["<$20k", "$20-40k", "$40-60k", "$60-80k", "$80-100k", ">$100k"]
income["income"] = pd.Categorical(income["income"], categories=order, ordered=True)
income.head()

,respondent_id,income
16,1000017,<$20k
23,1000024,<$20k
24,1000025,<$20k
28,1000029,<$20k
35,1000036,<$20k


## 5. Sex and age group in the same header: `hh_<sex>_<age>`  ->  `sex`, `age_group`, `members`

In [6]:
hh_cols = [c for c in df.columns if c.startswith("hh_")]

household_members = df.melt(
    id_vars="respondent_id",
    value_vars=hh_cols,
    var_name="sex_age",
    value_name="members",
).dropna(subset=["members"])

# 'hh_m_0_14' -> remove 'hh_', then split once on the first underscore
parts = household_members["sex_age"].str.replace("hh_", "", regex=False).str.split("_", n=1, expand=True)
household_members["sex"] = parts[0].map({"m": "male", "f": "female"})
household_members["age_group"] = parts[1].str.replace("_", "-").str.replace("plus", "+")
household_members["members"] = household_members["members"].astype(int)
household_members = (household_members.drop(columns="sex_age")
                     .sort_values(["respondent_id", "sex", "age_group"]).reset_index(drop=True))
print(household_members.shape)
household_members.head()

(424875, 4)


,respondent_id,members,sex,age_group
0,1000001,0,female,0-14
1,1000001,1,female,15-24
2,1000001,0,female,25-44
3,1000001,0,female,45-64
4,1000001,0,female,65+


## 6. Respondent table (identifier + respondent-level attributes)

In [7]:
respondents = (df[["respondent_id", "province", "interview_date"]]
               .merge(income, on="respondent_id"))
respondents["interview_date"] = pd.to_datetime(respondents["interview_date"])
print(respondents.shape)
respondents.head()

(50000, 4)


,respondent_id,province,interview_date,income
0,1000001,QC,2024-05-09,$40-60k
1,1000002,BC,2024-08-03,$60-80k
2,1000003,QC,2024-05-31,$40-60k
3,1000004,ON,2024-12-30,$40-60k
4,1000005,ON,2024-02-27,$80-100k


## 7. Check and save

In [8]:
tables = {
    "respondents": respondents,
    "weekly_spending": weekly_spending,
    "satisfaction": satisfaction,
    "household_members": household_members,
}
for name, t in tables.items():
    print(f"{name:20s} {t.shape}  missing values: {int(t.isna().sum().sum())}")

# No data lost: counts of non-missing values match the wide file
assert len(weekly_spending) == df[wk_cols].notna().sum().sum()
assert len(household_members) == df[hh_cols].notna().sum().sum()

import os
os.makedirs("data/tidy", exist_ok=True)
for name, t in tables.items():
    t.to_csv(f"data/tidy/{name}.csv", index=False)

respondents          (50000, 4)  missing values: 0
weekly_spending      (779314, 3)  missing values: 0
satisfaction         (237630, 4)  missing values: 7199
household_members    (424875, 4)  missing values: 0


## Summary

- `province` standardised (`" ON "`, `"on"` -> `"ON"`).
- Four headers-as-values problems melted into `week`/`spend`, `item`/`response`, `income`, and `sex`/`age_group`/`members`.
- Missing weekly and household cells dropped (a missing cell is the absence of an observation, not a value).
- Refusal code 7 kept as a flag and replaced with a missing rating.
- Each table holds one observational unit and joins back on `respondent_id`.